# MLflow Runs Analyzer (reproducibility tables + plots)
Builds aggregated tables (Baseline Neutral, Baseline Fair, FACTER iter=3) per model/dataset/mode from MLflow runs.

## Setup

### General Imports and Config

In [1]:
import os
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)

OUTPUT_DIR = Path("mlflow_analysis_output")
OUTPUT_DIR.mkdir(exist_ok=True)

# CONFIG
EXPERIMENT_NAME = "facter-repro"
ITER_REPORT = 3                        # iteration to report on
INCLUDE_PREDICT_MODES = None           # {"open","rank"} or None for all
INCLUDE_DATASETS = None                # {"ml-1m","amazon"} or None for all
INCLUDE_MODELS_SUBSTR = None           # {"llama-3", "mistral"} or None for all
TRACKING_DB = Path("mlflow.db")
TRACKING_MLRUNS = Path("mlruns")

### MLflow & Load Runs

In [2]:
import mlflow

def resolve_tracking_uri() -> str:
    """
    Prefer sqlite tracking if mlflow.db exists; otherwise fall back to file-store mlruns.
    """
    if TRACKING_DB.exists():
        return f"sqlite:///{TRACKING_DB.resolve()}"
    if TRACKING_MLRUNS.exists():
        return str(TRACKING_MLRUNS.resolve())
    raise FileNotFoundError(
        f"Could not find tracking store. Expected {TRACKING_DB} or {TRACKING_MLRUNS}."
    )

tracking_uri = resolve_tracking_uri()
mlflow.set_tracking_uri(tracking_uri)
print("Tracking URI:", tracking_uri)

exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if exp is None:
    raise ValueError(f"MLflow experiment not found: {EXPERIMENT_NAME}")

print("Experiment:", exp.name, "| id:", exp.experiment_id)

# Pull all runs in this experiment (pandas DataFrame).
runs_df = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string='attributes.status = "FINISHED"',
    output_format="pandas",
)

print("Loaded runs:", len(runs_df))
runs_df.head(3)

Tracking URI: sqlite:////home/ozzy/Desktop/facter-repr/mlflow.db


2026/01/28 21:40:08 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/01/28 21:40:08 INFO mlflow.store.db.utils: Updating database tables
2026/01/28 21:40:08 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/28 21:40:08 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/01/28 21:40:08 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/28 21:40:08 INFO alembic.runtime.migration: Will assume non-transactional DDL.


Experiment: facter-repro | id: 1
Loaded runs: 28


,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.iter2.SNSV,metrics.baseline_neutral.SNSR.gender,metrics.baseline_fair.NDCG_10,metrics.baseline_neutral.n_violations,...,tags.git.commit,tags.mlflow.user,tags.platform,tags.dataset,tags.model_id,tags.mlflow.runName,tags.python,tags.mlflow.source.type,tags.mlflow.source.name,tags.git.state
0,0266519be494477c8bac7b275444342c,1,FINISHED,file:///gpfs/home4/scur0063/facter-repr/mlruns...,2026-01-28 15:26:41.243000+00:00,2026-01-28 17:49:33.818000+00:00,0.000000,0.023009,0.004546,108.0,...,89db3135e1203a4b97ecfdc94cd7623bc78fa28e,scur0063,Linux-5.14.0-570.76.1.el9_6.x86_64-x86_64-with...,amazon,meta-llama/Meta-Llama-3-8B-Instruct,amazon_meta-llama/Meta-Llama-3-8B-Instruct_see...,3.11.14,LOCAL,scripts/run_facter.py,dirty
1,88667f8b7aaa4ee4aacca2a19911a32b,1,FINISHED,file:///gpfs/home4/scur0063/facter-repr/mlruns...,2026-01-28 12:43:57.560000+00:00,2026-01-28 12:59:42.209000+00:00,0.019741,0.004050,0.454975,60.0,...,89db3135e1203a4b97ecfdc94cd7623bc78fa28e,scur0063,Linux-5.14.0-570.76.1.el9_6.x86_64-x86_64-with...,ml-1m,meta-llama/Meta-Llama-3-8B-Instruct,ml-1m_meta-llama/Meta-Llama-3-8B-Instruct_seed...,3.11.14,LOCAL,scripts/run_facter.py,dirty
2,927cf28d77a94d95a9ac828e2fc8b730,1,FINISHED,file:///gpfs/home4/scur0063/facter-repr/mlruns...,2026-01-28 12:18:28.048000+00:00,2026-01-28 15:43:04.184000+00:00,0.026805,0.024676,0.032747,46.0,...,89db3135e1203a4b97ecfdc94cd7623bc78fa28e,scur0063,Linux-5.14.0-570.76.1.el9_6.x86_64-x86_64-with...,ml-1m,meta-llama/Meta-Llama-3-8B-Instruct,ml-1m_meta-llama/Meta-Llama-3-8B-Instruct_seed...,3.11.14,LOCAL,scripts/run_facter.py,dirty
